In [2]:
"""
Step 3: proper Granger causality test, fixing the shared-trend AND
shared-weekly-pattern problems found in the raw lag scan.

Workflow:
  1. DESEASONALIZE: regress each series on day-of-week dummies (+ a
     Patch-Tuesday indicator, since CVE disclosures cluster on the second
     Tuesday of the month) and keep the residuals. Differencing alone does
     NOT remove a repeating weekly pattern -- it only removes a straight-line
     trend -- so a shared weekly rhythm (Reddit posting dips on weekends;
     CVE disclosure clusters mid-week/Patch Tuesday) can survive the trend
     fix and still produce spurious "significance" in both directions.
  2. ADF unit-root test PLUS an explicit linear-trend test on the
     deseasonalized residuals. A series can pass the unit-root test while
     still having a real slope over time, so both checks run independently.
  3. If EITHER check flags a problem, difference BOTH series in a pair by
     the same order.
  4. VAR order selection (AIC and BIC) picks a defensible lag instead of
     scanning every lag and cherry-picking.
  5. Granger causality test at both selected lags, in BOTH directions:
       - does CVE volume Granger-cause exit-intention volume? (the hypothesis)
       - does exit-intention volume Granger-cause CVE volume? (should NOT be
         significant -- if it is, something shared is still contaminating
         both series)

Reads the CSV from build_q1_timeseries.py. No raw file access here.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
import os
import warnings

warnings.filterwarnings("ignore")  # statsmodels VAR is chatty about lag-order edge cases

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q2_analysis/"
DAILY_CSV = os.path.join(OUT_DIR, "exit_daily_series.csv")

EXIT_COLS = ["n_exit_any", "n_exit_explicit"]
CVE_COLS = ["n_cve_run1", "n_cve_run2"]

MAX_LAG_FOR_ORDER_SELECTION = 84
MAX_DIFF_ORDER = 2
ALPHA = 0.05


def is_patch_tuesday(date):
    """Second Tuesday of the month -- the day vendors (notably Microsoft)
    routinely release patches and disclose vulnerabilities, a well-known
    source of CVE-publication clustering."""
    return 1 if (date.weekday() == 1 and 8 <= date.day <= 14) else 0


def deseasonalize(daily_df, date_col, value_cols):
    """Regress each value column on day-of-week dummies + Patch-Tuesday
    indicator. Returns a DataFrame of residuals (same index/order as input)
    plus prints the R^2 for each column so you can see how much weekly
    pattern was actually present."""
    dates = pd.to_datetime(daily_df[date_col])
    dow = dates.dt.dayofweek
    patch_tues = dates.apply(is_patch_tuesday)

    dow_dummies = pd.get_dummies(dow, prefix="dow", drop_first=True)
    X = pd.concat([dow_dummies, patch_tues.rename("patch_tuesday")], axis=1)
    X = sm.add_constant(X).astype(float)

    residuals = {}
    print("  Day-of-week + Patch-Tuesday deseasonalization:")
    for col in value_cols:
        y = daily_df[col].astype(float)
        model = sm.OLS(y, X).fit()
        residuals[col] = model.resid
        print(f"    {col}: R^2 = {model.rsquared:.4f} "
              f"({'meaningful weekly pattern present' if model.rsquared > 0.02 else 'little/no weekly pattern'})")

    return pd.DataFrame(residuals)


def adf_pvalue(series):
    if series.std() == 0:
        return 1.0  # constant series -- treat as "non-stationary" (forces a difference attempt,
                     # which will surface a clearer error if the series is genuinely too sparse)
    return adfuller(series.dropna(), autolag="AIC")[1]


def trend_pvalue_and_slope(series):
    """OLS regression of the series against a plain time index (0,1,2,...).
    Returns (p_value, slope) for the slope coefficient. A significant
    p-value means a real deterministic trend exists over the period, even
    if the ADF unit-root test alone doesn't flag it."""
    s = series.dropna().values
    t = np.arange(len(s))
    slope, intercept, r, p, se = scipy_stats.linregress(t, s)
    return p, slope


def find_common_diff_order(series_a, series_b, label_a, label_b, max_d=MAX_DIFF_ORDER):
    """Difference both series together, by the same number of steps, until
    BOTH pass the unit-root test AND show no significant linear trend (or
    until max_d is hit)."""
    a, b = series_a.copy(), series_b.copy()
    d = 0
    while d <= max_d:
        p_a_unit, p_b_unit = adf_pvalue(a), adf_pvalue(b)
        p_a_trend, slope_a = trend_pvalue_and_slope(a)
        p_b_trend, slope_b = trend_pvalue_and_slope(b)

        unit_a = "no unit root" if p_a_unit < ALPHA else "UNIT ROOT"
        unit_b = "no unit root" if p_b_unit < ALPHA else "UNIT ROOT"
        trend_a = f"TREND (slope={slope_a:.5f}, p={p_a_trend:.4g})" if p_a_trend < ALPHA else "no significant trend"
        trend_b = f"TREND (slope={slope_b:.5f}, p={p_b_trend:.4g})" if p_b_trend < ALPHA else "no significant trend"

        print(f"    d={d}: {label_a}: ADF p={p_a_unit:.4g} ({unit_a}); linear trend check: {trend_a}")
        print(f"           {label_b}: ADF p={p_b_unit:.4g} ({unit_b}); linear trend check: {trend_b}")

        a_ok = (p_a_unit < ALPHA) and (p_a_trend >= ALPHA)
        b_ok = (p_b_unit < ALPHA) and (p_b_trend >= ALPHA)

        if a_ok and b_ok:
            return a, b, d

        a = a.diff().dropna()
        b = b.diff().dropna()
        d += 1
    print(f"    WARNING: still non-stationary/trending after d={max_d} -- proceeding anyway, "
          f"interpret results cautiously")
    return a, b, d


def select_lag_orders(df_pair, maxlags):
    model = VAR(df_pair)
    sel = model.select_order(maxlags=maxlags)
    return sel


def granger_at_lag(effect_series, cause_series, lag, direction_label):
    """data columns must be [effect, cause] -- statsmodels tests whether
    column 1 (cause) Granger-causes column 0 (effect)."""
    df_pair = pd.DataFrame({"effect": effect_series.values, "cause": cause_series.values})
    results = grangercausalitytests(df_pair, maxlag=[lag], verbose=False)
    fstat, pvalue, df_denom, df_num = results[lag][0]["ssr_ftest"]
    sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"
    print(f"    {direction_label} at lag={lag}: F={fstat:.4f}, p={pvalue:.4g}  -> {sig} (alpha={ALPHA})")
    return fstat, pvalue


def main():
    if not os.path.exists(DAILY_CSV):
        print(f"Could not find {DAILY_CSV}. Run build_q1_timeseries.py first.")
        return

    daily = pd.read_csv(DAILY_CSV)
    print(f"Loaded {len(daily)} days from {DAILY_CSV}")

    print(f"\n{'=' * 70}")
    print("DESEASONALIZING (day-of-week + Patch Tuesday)")
    print("=" * 70)
    deseasonalized = deseasonalize(daily, "date", EXIT_COLS + CVE_COLS)

    summary_rows = []

    for exit_col in EXIT_COLS:
        for cve_col in CVE_COLS:
            print(f"\n{'=' * 70}")
            print(f"{exit_col}  vs  {cve_col}")
            print("=" * 70)

            print("\n  Stationarity check + common differencing order (on deseasonalized residuals):")
            exit_s, cve_s, d = find_common_diff_order(
                deseasonalized[exit_col], deseasonalized[cve_col], exit_col, cve_col
            )
            print(f"  Using d={d} for both series in this pair")

            df_pair = pd.DataFrame({exit_col: exit_s.values, cve_col: cve_s.values})

            print(f"\n  Selecting lag order (AIC/BIC, up to {MAX_LAG_FOR_ORDER_SELECTION} days):")
            sel = select_lag_orders(df_pair, MAX_LAG_FOR_ORDER_SELECTION)
            aic_lag = max(int(sel.aic), 1)
            bic_lag = max(int(sel.bic), 1)
            print(f"  AIC-selected lag: {aic_lag}   |   BIC-selected lag: {bic_lag}")

            print(f"\n  Granger causality -- does CVE lead exit intention?")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(exit_s, cve_s, lag, f"CVE -> exit intention ({crit_name} lag)")
                summary_rows.append({
                    "exit_threshold": exit_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "cve_to_exit", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

            print(f"\n  Reverse check -- does exit intention lead CVE? (should NOT be significant for a clean story)")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(cve_s, exit_s, lag, f"exit intention -> CVE ({crit_name} lag)")
                summary_rows.append({
                    "exit_threshold": exit_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "exit_to_cve", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

    summary_df = pd.DataFrame(summary_rows)
    out_path = os.path.join(OUT_DIR, "exit_granger_causality_results.csv")
    summary_df.to_csv(out_path, index=False)

    print(f"\n{'=' * 70}")
    print("FULL SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print(f"\nSaved -> {out_path}")

    print(f"\n{'=' * 70}")
    print("HOW TO READ THIS")
    print("=" * 70)
    print("The bar for a real finding: direction='cve_to_exit' significant (p < 0.05)")
    print("AND the matching direction='exit_to_cve' row (same combination + criterion)")
    print("NOT significant. Only that combination counts as clean evidence that CVE")
    print("volume helps predict exit-intention volume, and not the other way around.")
    print("\nIf exit_to_cve is significant ANYWHERE, that specific row is not trustworthy")
    print("evidence for your hypothesis, regardless of what cve_to_exit shows for the same")
    print("combination -- it means something is still shared between the two series that")
    print("hasn't been fully removed (weekly pattern, trend, or some other confound).")
    print("\nWeight BIC rows more than AIC rows when they disagree -- AIC tends to pick")
    print("larger, more overfit-prone lags; BIC's smaller, more conservative lag is the")
    print("sturdier basis for a real claim.")


if __name__ == "__main__":
    main()

Loaded 3071 days from /Users/nadia/Desktop/redditRun_june/q2_analysis/exit_daily_series.csv

DESEASONALIZING (day-of-week + Patch Tuesday)
  Day-of-week + Patch-Tuesday deseasonalization:
    n_exit_any: R^2 = 0.1063 (meaningful weekly pattern present)
    n_exit_explicit: R^2 = 0.0197 (little/no weekly pattern)
    n_cve_run1: R^2 = 0.1317 (meaningful weekly pattern present)
    n_cve_run2: R^2 = 0.0330 (meaningful weekly pattern present)

n_exit_any  vs  n_cve_run1

  Stationarity check + common differencing order (on deseasonalized residuals):
    d=0: n_exit_any: ADF p=0.009129 (no unit root); linear trend check: TREND (slope=0.00124, p=9.801e-157)
           n_cve_run1: ADF p=1.02e-08 (no unit root); linear trend check: TREND (slope=-0.00128, p=3.676e-16)
    d=1: n_exit_any: ADF p=5.621e-30 (no unit root); linear trend check: no significant trend
           n_cve_run1: ADF p=1.009e-29 (no unit root); linear trend check: no significant trend
  Using d=1 for both series in this pai